In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import re
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
try:
    df_train_price = pd.read_csv('ruc_Class25Q2_train_price.csv')
    
    df_test_price = pd.read_csv('ruc_Class25Q2_test_price.csv')
   

    print(f"成功加载 ruc_Class25Q2_train_price.csv，维度: {df_train_price.shape}")
   
    print(f"成功加载 ruc_Class25Q2_test_price.csv， 维度: {df_test_price.shape}")
   
except FileNotFoundError:
    print("错误")

# 房价数据预测
# 查看数据前五行
print("\n--- 训练集前5行 ---")
print(df_train_price.head())

# 查看数据的摘要
print("\n--- 训练集信息 ---")
df_train_price.info()

# 查看数值型数据的基本统计
print("\n--- 训练集数值统计 ---")
print(df_train_price.describe())

C:\Users\wangy\AppData\Local\Temp\ipykernel_28504\183080258.py:2: DtypeWarning: Columns (3,32,34,43,46,49,51) have mixed types. Specify dtype option on import or set low_memory=False.
  df_train_price = pd.read_csv('ruc_Class25Q2_train_price.csv')
C:\Users\wangy\AppData\Local\Temp\ipykernel_28504\183080258.py:4: DtypeWarning: Columns (4,32) have mixed types. Specify dtype option on import or set low_memory=False.
  df_test_price = pd.read_csv('ruc_Class25Q2_test_price.csv')


成功加载 ruc_Class25Q2_train_price.csv，维度: (103871, 55)
成功加载 ruc_Class25Q2_test_price.csv， 维度: (34017, 55)

--- 训练集前5行 ---
   城市     区域      板块    环线         Price      房屋户型        所在楼层     建筑面积  \
0   0  109.0   150.0  二至三环  6.194049e+06  2室1厅1厨1卫   中楼层 (共5层)    52.3㎡   
1   0   65.0   299.0  五至六环  4.354153e+06  3室1厅1厨1卫    顶层 (共6层)  127.44㎡   
2   0   62.0   911.0  五至六环  3.321992e+06  3室2厅1厨2卫   低楼层 (共6层)  118.02㎡   
3   0  123.0  1102.0   六环外  7.895656e+06  6室3厅1厨3卫    底层 (共2层)  293.23㎡   
4   0   81.0   295.0  三至四环  1.902960e+06     1房间1卫  中楼层 (共10层)   39.85㎡   

      套内面积     房屋朝向  ...     供水        供暖     供电            燃气费       供热费  \
0      NaN      南 北  ...     民水      集中供暖     民电       2.61元/m³     30元/㎡   
1   123.7㎡      南 北  ...  商水/民水       自采暖  商电/民电       2.61元/m³       NaN   
2  101.95㎡       东南  ...  商水/民水  集中供暖/自采暖  商电/民电       2.61元/m³     30元/㎡   
3  293.23㎡  东 南 西 北  ...     民水       自采暖     民电  2.61-2.63元/m³       NaN   
4   29.94㎡        南  ...  商水/民水  集中供暖/自采暖  商电

开始数据处理（删除非特征或无关变量，删除缺失率过高的变量）

In [3]:
# 定义要删除的列名
columns_to_drop = [
    '客户反馈',  # 1. 数据泄漏
    '抵押信息',  # 2. 完全无用 (100% 缺失)
    '别墅类型',  # 3. 信息过少 (98.6% 缺失)
    '物业办公电话', # 4. 非特征 (ID)
    
    # 5. 缺失率过高 (>70%) 且难以填充的列
    '户型介绍',
    '环线位置',
    '供暖',
    '供热费' 
]

# 删除这些列
df_train_price_cleaned = df_train_price.drop(columns=columns_to_drop).copy()

# 测试集上删除同样的列，保证训练集和测试集有相同的特征
try:
    df_test_price_cleaned = df_test_price.drop(columns=columns_to_drop).copy()
    
except NameError:
    print("错误")

In [4]:

# 基础数据清洗
print("\n--- 步骤3: 基础数据清洗 ---")
df_train_price_cleaned['建筑面积'] = df_train_price_cleaned['建筑面积'].str.replace('㎡', '').astype(float)
df_test_price_cleaned['建筑面积'] = df_test_price_cleaned['建筑面积'].str.replace('㎡', '').astype(float)
df_train_price_cleaned['套内面积'] = df_train_price_cleaned['套内面积'].str.replace('㎡', '').astype(float)
df_test_price_cleaned['套内面积'] = df_test_price_cleaned['套内面积'].str.replace('㎡', '').astype(float)
print("已完成建筑面积和套内面积的单位清理")



--- 步骤3: 基础数据清洗 ---
已完成建筑面积和套内面积的单位清理


In [5]:

# 分离特征和目标变量
print("\n--- 步骤4: 数据分割 ---")
y = df_train_price_cleaned['Price']
X = df_train_price_cleaned.drop(columns=['Price'])

# 数据分割
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=111)
X_train = X_train.copy()
X_val = X_val.copy()
X_test_price = df_test_price_cleaned.copy()

if 'ID' in X_test_price.columns:
    X_test_price = X_test_price.drop(columns=['ID'])

print(f"训练集维度: {X_train.shape}")
print(f"验证集维度: {X_val.shape}")
print(f"测试集维度: {X_test_price.shape}")



--- 步骤4: 数据分割 ---
训练集维度: (83096, 46)
验证集维度: (20775, 46)
测试集维度: (34017, 46)


In [6]:

# 缺失值填充
print("\n--- 步骤5: 缺失值填充 ---")
numerical_features = X_train.select_dtypes(include=['float64', 'int64']).columns
categorical_features = X_train.select_dtypes(include=['object']).columns

print(f"数值型特征: {len(numerical_features)}个，使用中位数填充")
print(f"类别型特征: {len(categorical_features)}个，使用'Unknown'填充")

for col in numerical_features:
    median_value = X_train[col].median()
    X_train[col] = X_train[col].fillna(median_value)
    X_val[col] = X_val[col].fillna(median_value)
    if col in X_test_price.columns:
        X_test_price[col] = X_test_price[col].fillna(median_value)

fill_value = "Unknown"
for col in categorical_features:
    X_train[col] = X_train[col].fillna(fill_value)
    X_val[col] = X_val[col].fillna(fill_value)
    if col in X_test_price.columns:
        X_test_price[col] = X_test_price[col].fillna(fill_value)

print("缺失值填充完成")



--- 步骤5: 缺失值填充 ---
数值型特征: 14个，使用中位数填充
类别型特征: 32个，使用'Unknown'填充
缺失值填充完成


In [7]:

# 特征工程 - 二元特征
print("\n--- 步骤6: 特征工程开始 ---")
print("--- 6.1: 创建二元特征 ---")

text_to_binary_cols = ['交通出行', '周边配套', '核心卖点']
for col in text_to_binary_cols:
    new_col_name = f'Has_{col}'
    X_train[new_col_name] = X_train[col].apply(lambda x: 1 if x != 'Unknown' else 0)
    X_val[new_col_name] = X_val[col].apply(lambda x: 1 if x != 'Unknown' else 0)
    X_test_price[new_col_name] = X_test_price[col].apply(lambda x: 1 if x != 'Unknown' else 0)

print(f"创建二元特征: {len(text_to_binary_cols)}个")



--- 步骤6: 特征工程开始 ---
--- 6.1: 创建二元特征 ---
创建二元特征: 3个


In [8]:

# 建筑年代处理
print("\n--- 6.2: 建筑年代处理 ---")
def parse_build_year(text_data):
    if not isinstance(text_data, str):
        return np.nan
    numbers = re.findall(r'(\d{4})', text_data)
    if len(numbers) == 0:
        return np.nan
    elif len(numbers) == 1:
        return float(numbers[0])
    else:
        return (float(numbers[0]) + float(numbers[-1])) / 2

year_extracted_train = X_train['建筑年代'].apply(parse_build_year)
year_extracted_val = X_val['建筑年代'].apply(parse_build_year)
year_extracted_test = X_test_price['建筑年代'].apply(parse_build_year)

X_train['House_Age'] = 2025 - year_extracted_train
X_val['House_Age'] = 2025 - year_extracted_val
X_test_price['House_Age'] = 2025 - year_extracted_test

median_age = X_train['House_Age'].median()
X_train['House_Age'] = X_train['House_Age'].fillna(median_age)
X_val['House_Age'] = X_val['House_Age'].fillna(median_age)
X_test_price['House_Age'] = X_test_price['House_Age'].fillna(median_age)

print("创建特征: House_Age (房龄)")



--- 6.2: 建筑年代处理 ---
创建特征: House_Age (房龄)


In [9]:

# 物业费处理
print("\n--- 6.3: 物业费处理 ---")
def parse_fee_simple(text):
    if not isinstance(text, str) or text == 'Unknown':
        return np.nan
    numbers = re.findall(r'(\d+\.?\d*)', text)
    numbers = [float(n) for n in numbers if n]
    return np.mean(numbers) if numbers else np.nan

def is_interval(text):
    if not isinstance(text, str) or text == 'Unknown':
        return 0
    return 1 if '-' in text else 0

X_train['Property_Fee'] = X_train['物 业 费'].apply(parse_fee_simple)
X_train['Property_Fee_IsInterval'] = X_train['物 业 费'].apply(is_interval)
X_val['Property_Fee'] = X_val['物 业 费'].apply(parse_fee_simple)
X_val['Property_Fee_IsInterval'] = X_val['物 业 费'].apply(is_interval)
X_test_price['Property_Fee'] = X_test_price['物 业 费'].apply(parse_fee_simple)
X_test_price['Property_Fee_IsInterval'] = X_test_price['物 业 费'].apply(is_interval)

median_fee = X_train['Property_Fee'].median()
for df in [X_train, X_val, X_test_price]:
    df['Property_Fee'] = df['Property_Fee'].fillna(median_fee)

print("创建特征: Property_Fee, Property_Fee_IsInterval")



--- 6.3: 物业费处理 ---
创建特征: Property_Fee, Property_Fee_IsInterval


In [10]:

# 房屋总数处理
print("\n--- 6.4: 房屋总数处理 ---")
def parse_total_units(text_data):
    if not isinstance(text_data, str):
        return np.nan
    match = re.search(r'(\d+)', text_data)
    return float(match.group(1)) if match else np.nan

X_train['Total_Units'] = X_train['房屋总数'].apply(parse_total_units)
X_val['Total_Units'] = X_val['房屋总数'].apply(parse_total_units)
X_test_price['Total_Units'] = X_test_price['房屋总数'].apply(parse_total_units)

median_units = X_train['Total_Units'].median()
X_train['Total_Units'] = X_train['Total_Units'].fillna(median_units)
X_val['Total_Units'] = X_val['Total_Units'].fillna(median_units)
X_test_price['Total_Units'] = X_test_price['Total_Units'].fillna(median_units)

print("创建特征: Total_Units")



--- 6.4: 房屋总数处理 ---
创建特征: Total_Units


In [11]:

# 交易时间处理
print("\n--- 6.5: 交易时间处理 ---")
train_dates = pd.to_datetime(X_train['交易时间'], errors='coerce')
val_dates = pd.to_datetime(X_val['交易时间'], errors='coerce')
test_dates = pd.to_datetime(X_test_price['交易时间'], errors='coerce')

X_train['Transaction_Year'] = train_dates.dt.year
X_train['Transaction_Month'] = train_dates.dt.month
X_val['Transaction_Year'] = val_dates.dt.year
X_val['Transaction_Month'] = val_dates.dt.month
X_test_price['Transaction_Year'] = test_dates.dt.year
X_test_price['Transaction_Month'] = test_dates.dt.month

last_trans_year_train = pd.to_datetime(X_train['上次交易'], errors='coerce').dt.year
last_trans_year_val = pd.to_datetime(X_val['上次交易'], errors='coerce').dt.year
last_trans_year_test = pd.to_datetime(X_test_price['上次交易'], errors='coerce').dt.year

X_train['Years_Since_Last_Transaction'] = X_train['Transaction_Year'] - last_trans_year_train
X_val['Years_Since_Last_Transaction'] = X_val['Transaction_Year'] - last_trans_year_val
X_test_price['Years_Since_Last_Transaction'] = X_test_price['Transaction_Year'] - last_trans_year_test

for col in ['Transaction_Year', 'Transaction_Month']:
    median_val = X_train[col].median()
    X_train[col] = X_train[col].fillna(median_val)
    X_val[col] = X_val[col].fillna(median_val)
    X_test_price[col] = X_test_price[col].fillna(median_val)

X_train['Years_Since_Last_Transaction'] = X_train['Years_Since_Last_Transaction'].fillna(0)
X_val['Years_Since_Last_Transaction'] = X_val['Years_Since_Last_Transaction'].fillna(0)
X_test_price['Years_Since_Last_Transaction'] = X_test_price['Years_Since_Last_Transaction'].fillna(0)

print("创建时间特征: Transaction_Year, Transaction_Month, Years_Since_Last_Transaction")



--- 6.5: 交易时间处理 ---


C:\Users\wangy\AppData\Local\Temp\ipykernel_28504\2639610080.py:14: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  last_trans_year_train = pd.to_datetime(X_train['上次交易'], errors='coerce').dt.year


创建时间特征: Transaction_Year, Transaction_Month, Years_Since_Last_Transaction


In [12]:

# 房屋户型处理
print("\n--- 6.6: 房屋户型处理 ---")
def parse_layout(text_data):
    if not isinstance(text_data, str):
        return (np.nan, np.nan, np.nan, np.nan)

    room_match = re.search(r'(\d+)(?:室|房间)', text_data)
    num_rooms = int(room_match.group(1)) if room_match else 0
    
    living_match = re.search(r'(\d+)厅', text_data)
    num_living = int(living_match.group(1)) if living_match else 0
    
    kitchen_match = re.search(r'(\d+)厨', text_data)
    num_kitchen = int(kitchen_match.group(1)) if kitchen_match else 0
    
    bath_match = re.search(r'(\d+)卫', text_data)
    num_bath = int(bath_match.group(1)) if bath_match else 0

    if num_rooms == 0 and num_living == 0 and num_kitchen == 0 and num_bath == 0:
         return (np.nan, np.nan, np.nan, np.nan)
    
    return (num_rooms, num_living, num_kitchen, num_bath)

new_cols_train = X_train['房屋户型'].apply(lambda x: pd.Series(parse_layout(x), index=['Num_Rooms', 'Num_LivingRooms', 'Num_Kitchens', 'Num_Baths']))
new_cols_val = X_val['房屋户型'].apply(lambda x: pd.Series(parse_layout(x), index=['Num_Rooms', 'Num_LivingRooms', 'Num_Kitchens', 'Num_Baths']))
new_cols_test = X_test_price['房屋户型'].apply(lambda x: pd.Series(parse_layout(x), index=['Num_Rooms', 'Num_LivingRooms', 'Num_Kitchens', 'Num_Baths']))

X_train = pd.concat([X_train, new_cols_train], axis=1)
X_val = pd.concat([X_val, new_cols_val], axis=1)
X_test_price = pd.concat([X_test_price, new_cols_test], axis=1)

new_layout_cols = ['Num_Rooms', 'Num_LivingRooms', 'Num_Kitchens', 'Num_Baths']
for col in new_layout_cols:
    median_val = X_train[col].median()
    X_train[col] = X_train[col].fillna(median_val)
    X_val[col] = X_val[col].fillna(median_val)
    X_test_price[col] = X_test_price[col].fillna(median_val)

print("创建户型特征: Num_Rooms, Num_LivingRooms, Num_Kitchens, Num_Baths")



--- 6.6: 房屋户型处理 ---
创建户型特征: Num_Rooms, Num_LivingRooms, Num_Kitchens, Num_Baths


In [13]:

# 所在楼层处理
print("\n--- 6.7: 所在楼层处理 ---")
def parse_total_floors(text_data):
    if not isinstance(text_data, str): 
        return np.nan
    match = re.search(r'共(\d+)层', text_data)
    return float(match.group(1)) if match else np.nan

X_train['Total_Floors'] = X_train['所在楼层'].apply(parse_total_floors)
X_val['Total_Floors'] = X_val['所在楼层'].apply(parse_total_floors)
X_test_price['Total_Floors'] = X_test_price['所在楼层'].apply(parse_total_floors)

median_total_floors = X_train['Total_Floors'].median()
X_train['Total_Floors'] = X_train['Total_Floors'].fillna(median_total_floors)
X_val['Total_Floors'] = X_val['Total_Floors'].fillna(median_total_floors)
X_test_price['Total_Floors'] = X_test_price['Total_Floors'].fillna(median_total_floors)

def calculate_floor_ratio(row):
    text_data = row['所在楼层']
    total_floors = row['Total_Floors']
    
    if not isinstance(text_data, str):
        return np.nan
    
    if '地下室' in text_data:
        return 0.0
    elif '底层' in text_data:
        return 1.0 / total_floors
    elif '顶层' in text_data:
        return 1.0
    elif '高楼层' in text_data:
        return 0.75
    elif '中楼层' in text_data:
        return 0.5
    elif '低楼层' in text_data:
        return 0.25
    else:
        return np.nan

X_train['Floor_Ratio'] = X_train.apply(calculate_floor_ratio, axis=1)
X_val['Floor_Ratio'] = X_val.apply(calculate_floor_ratio, axis=1)
X_test_price['Floor_Ratio'] = X_test_price.apply(calculate_floor_ratio, axis=1)

median_floor_ratio = X_train['Floor_Ratio'].median()
X_train['Floor_Ratio'] = X_train['Floor_Ratio'].fillna(median_floor_ratio)
X_val['Floor_Ratio'] = X_val['Floor_Ratio'].fillna(median_floor_ratio)
X_test_price['Floor_Ratio'] = X_test_price['Floor_Ratio'].fillna(median_floor_ratio)

X_train['Estimated_Floor'] = X_train['Total_Floors'] * X_train['Floor_Ratio']
X_val['Estimated_Floor'] = X_val['Total_Floors'] * X_val['Floor_Ratio']
X_test_price['Estimated_Floor'] = X_test_price['Total_Floors'] * X_test_price['Floor_Ratio']

print("创建楼层特征: Floor_Ratio, Estimated_Floor")



--- 6.7: 所在楼层处理 ---
创建楼层特征: Floor_Ratio, Estimated_Floor


In [14]:
# 梯户比例处理
print("\n--- 6.8: 梯户比例处理 ---")
chinese_num_map_basic = {
    '一': 1, '两': 2, '三': 3, '四': 4, '五': 5, 
    '六': 6, '七': 7, '八': 8, '九': 9, '十': 10
}

# 扩展中文数字解析函数
def parse_extended_chinese_number(chinese_str):
    """
    解析扩展中文数字，支持十几、几十、几十几等格式
    """
    if not chinese_str:
        return np.nan
    
    # 如果是纯阿拉伯数字，直接转换
    if chinese_str.isdigit():
        return float(chinese_str)
    
    # 定义数字映射
    digit_map = {
        '零': 0, '一': 1, '二': 2, '两': 2, '三': 3, '四': 4, 
        '五': 5, '六': 6, '七': 7, '八': 8, '九': 9, '十': 10
    }
    
    # 处理各种中文数字格式
    if '十' in chinese_str:
        # 找到"十"的位置
        ten_index = chinese_str.index('十')
        
        if len(chinese_str) == 1:
            # 只有"十"
            return 10.0
        elif len(chinese_str) == 2:
            if ten_index == 0:
                # 十几 (如: 十一、十二...十九)
                ones_digit = digit_map.get(chinese_str[1], 0)
                return 10.0 + ones_digit
            else:
                # 几十 (如: 二十、三十...九十)
                tens_digit = digit_map.get(chinese_str[0], 0)
                return tens_digit * 10.0
        elif len(chinese_str) == 3:
            # 几十几 (如: 二十一、三十二...九十九)
            tens_digit = digit_map.get(chinese_str[0], 0)
            ones_digit = digit_map.get(chinese_str[2], 0)
            return tens_digit * 10.0 + ones_digit
    
    # 如果是单个基础中文字符，使用基础映射
    if chinese_str in chinese_num_map_basic:
        return float(chinese_num_map_basic[chinese_str])
    
    return np.nan

def parse_elevator_ratio_optimized(text_data):
    if not isinstance(text_data, str) or text_data == 'Unknown':
        return (np.nan, np.nan)
    
    # 提取电梯数
    elevator_match_num = re.search(r'(\d+)梯', text_data)
    if elevator_match_num:
        num_elevators = float(elevator_match_num.group(1))
    else:
        # 尝试匹配中文数字
        elevator_match_ext = re.search(r'([一二三四五六七八九十两零\d]+)梯', text_data)
        if elevator_match_ext:
            num_elevators = parse_extended_chinese_number(elevator_match_ext.group(1))
        else:
            num_elevators = np.nan

    # 提取住户数
    household_match_num = re.search(r'(\d+)户', text_data)
    if household_match_num:
        num_households = float(household_match_num.group(1))
    else:
        household_match_ext = re.search(r'([一二三四五六七八九十两零\d]+)户', text_data)
        if household_match_ext:
            num_households = parse_extended_chinese_number(household_match_ext.group(1))
        else:
            num_households = np.nan
            
    return (num_elevators, num_households)

# 测试各种情况
test_cases = [
    "十一梯20户",
    "二十梯三十户", 
    "二梯十户",
    "5梯20户",
    "三梯五户",
    "二十五梯四十户",
    "十梯十户",
    "三十梯五十户",
    "十五梯二十户",
    "十二梯八户"
]

print("测试梯户比例解析:")
for test_text in test_cases:
    result = parse_elevator_ratio_optimized(test_text)
    print(f"  '{test_text}' -> {result}")

# 验证扩展解析函数
print("\n验证扩展解析函数:")
test_numbers = ["十一", "二十", "二十五", "十", "十五", "十二"]
for num in test_numbers:
    result = parse_extended_chinese_number(num)
    print(f"  '{num}' -> {result}")

new_cols_train = X_train['梯户比例'].apply(
    lambda x: pd.Series(parse_elevator_ratio_optimized(x), 
                       index=['Num_Elevators', 'Num_Households']))
new_cols_val = X_val['梯户比例'].apply(
    lambda x: pd.Series(parse_elevator_ratio_optimized(x), 
                       index=['Num_Elevators', 'Num_Households']))
new_cols_test = X_test_price['梯户比例'].apply(
    lambda x: pd.Series(parse_elevator_ratio_optimized(x), 
                       index=['Num_Elevators', 'Num_Households']))

X_train = pd.concat([X_train, new_cols_train], axis=1)
X_val = pd.concat([X_val, new_cols_val], axis=1)
X_test_price = pd.concat([X_test_price, new_cols_test], axis=1)

print(f"创建梯户特征后的维度:")
print(f"X_train: {X_train.shape}")
print(f"X_val: {X_val.shape}")
print(f"X_test_price: {X_test_price.shape}")

new_elevator_cols = ['Num_Elevators', 'Num_Households']
for col in new_elevator_cols:
    median_val = X_train[col].median()
    X_train[col] = X_train[col].fillna(median_val)
    X_val[col] = X_val[col].fillna(median_val)
    X_test_price[col] = X_test_price[col].fillna(median_val)
    print(f"填充 {col} 的缺失值，使用中位数: {median_val}")

X_train['Elevator_Ratio'] = X_train['Num_Elevators'] / X_train['Num_Households'].clip(lower=1)
X_val['Elevator_Ratio'] = X_val['Num_Elevators'] / X_val['Num_Households'].clip(lower=1)
X_test_price['Elevator_Ratio'] = X_test_price['Num_Elevators'] / X_test_price['Num_Households'].clip(lower=1)

print("创建梯户特征: Num_Elevators, Num_Households, Elevator_Ratio")
print(f"梯户特征统计信息:")
print(f"Num_Elevators - 训练集均值: {X_train['Num_Elevators'].mean():.2f}, 中位数: {X_train['Num_Elevators'].median():.2f}")
print(f"Num_Households - 训练集均值: {X_train['Num_Households'].mean():.2f}, 中位数: {X_train['Num_Households'].median():.2f}")
print(f"Elevator_Ratio - 训练集均值: {X_train['Elevator_Ratio'].mean():.4f}, 中位数: {X_train['Elevator_Ratio'].median():.4f}")


--- 6.8: 梯户比例处理 ---
测试梯户比例解析:
  '十一梯20户' -> (11.0, 20.0)
  '二十梯三十户' -> (20.0, 30.0)
  '二梯十户' -> (nan, 10.0)
  '5梯20户' -> (5.0, 20.0)
  '三梯五户' -> (3.0, 5.0)
  '二十五梯四十户' -> (25.0, 40.0)
  '十梯十户' -> (10.0, 10.0)
  '三十梯五十户' -> (30.0, 50.0)
  '十五梯二十户' -> (15.0, 20.0)
  '十二梯八户' -> (12.0, 8.0)

验证扩展解析函数:
  '十一' -> 11.0
  '二十' -> 20.0
  '二十五' -> 25.0
  '十' -> 10.0
  '十五' -> 15.0
  '十二' -> 12.0
创建梯户特征后的维度:
X_train: (83096, 65)
X_val: (20775, 65)
X_test_price: (34017, 65)
填充 Num_Elevators 的缺失值，使用中位数: 2.0
填充 Num_Households 的缺失值，使用中位数: 4.0
创建梯户特征: Num_Elevators, Num_Households, Elevator_Ratio
梯户特征统计信息:
Num_Elevators - 训练集均值: 1.94, 中位数: 2.00
Num_Households - 训练集均值: 6.21, 中位数: 4.00
Elevator_Ratio - 训练集均值: 0.4362, 中位数: 0.4286


In [15]:

# 其他特征处理（绿化率、燃气费、楼栋总数）
print("\n--- 6.9: 其他特征处理 ---")
def parse_number_simple(text):
    if not isinstance(text, str) or text == 'Unknown':
        return np.nan
    numbers = re.findall(r'(\d+\.?\d*)', text)
    numbers = [float(n) for n in numbers if n]
    return np.mean(numbers) if numbers else np.nan



--- 6.9: 其他特征处理 ---


In [16]:

# 绿化率
X_train['Greening_Rate'] = X_train['绿 化 率'].apply(parse_number_simple) / 100.0
X_val['Greening_Rate'] = X_val['绿 化 率'].apply(parse_number_simple) / 100.0
X_test_price['Greening_Rate'] = X_test_price['绿 化 率'].apply(parse_number_simple) / 100.0
median_rate = X_train['Greening_Rate'].median()
X_train['Greening_Rate'] = X_train['Greening_Rate'].fillna(median_rate)
X_val['Greening_Rate'] = X_val['Greening_Rate'].fillna(median_rate)
X_test_price['Greening_Rate'] = X_test_price['Greening_Rate'].fillna(median_rate)

# 燃气费
X_train['Gas_Fee'] = X_train['燃气费'].apply(parse_number_simple)
X_val['Gas_Fee'] = X_val['燃气费'].apply(parse_number_simple)
X_test_price['Gas_Fee'] = X_test_price['燃气费'].apply(parse_number_simple)
median_gas_fee = X_train['Gas_Fee'].median()
X_train['Gas_Fee'] = X_train['Gas_Fee'].fillna(median_gas_fee)
X_val['Gas_Fee'] = X_val['Gas_Fee'].fillna(median_gas_fee)
X_test_price['Gas_Fee'] = X_test_price['Gas_Fee'].fillna(median_gas_fee)


# 楼栋总数
X_train['Total_Buildings'] = X_train['楼栋总数'].apply(parse_number_simple)
X_val['Total_Buildings'] = X_val['楼栋总数'].apply(parse_number_simple)
X_test_price['Total_Buildings'] = X_test_price['楼栋总数'].apply(parse_number_simple)
median_buildings = X_train['Total_Buildings'].median()
X_train['Total_Buildings'] = X_train['Total_Buildings'].fillna(median_buildings)
X_val['Total_Buildings'] = X_val['Total_Buildings'].fillna(median_buildings)
X_test_price['Total_Buildings'] = X_test_price['Total_Buildings'].fillna(median_buildings)


In [17]:

# 多项式特征
print("\n--- 6.10: 创建多项式特征 ---")
X_train['Area_sq'] = X_train['建筑面积'] ** 2
X_val['Area_sq'] = X_val['建筑面积'] ** 2
X_test_price['Area_sq'] = X_test_price['建筑面积'] ** 2
print("创建多项式特征: Area_sq")



--- 6.10: 创建多项式特征 ---
创建多项式特征: Area_sq


In [18]:

# 目标变量对数转换
print("\n--- 步骤7: 目标变量对数转换 ---")
y_train_log = np.log1p(y_train)
y_val_log = np.log1p(y_val)
print("目标变量对数转换完成")

# 新增特征：产权所属、房屋优势、配备电梯
print("\n--- 步骤8: 新增特征处理 ---")



--- 步骤7: 目标变量对数转换 ---
目标变量对数转换完成

--- 步骤8: 新增特征处理 ---


In [19]:

# 产权所属
def parse_property_rights(text):
    if not isinstance(text, str) or text == 'Unknown':
        return np.nan
    return 1 if text == '共有' else 0

X_train['Property_Rights_Shared'] = X_train['产权所属'].apply(parse_property_rights)
X_val['Property_Rights_Shared'] = X_val['产权所属'].apply(parse_property_rights)
X_test_price['Property_Rights_Shared'] = X_test_price['产权所属'].apply(parse_property_rights)
mode_rights = X_train['Property_Rights_Shared'].mode()[0]
for df in [X_train, X_val, X_test_price]:
    df['Property_Rights_Shared'] = df['Property_Rights_Shared'].fillna(mode_rights)


In [20]:

# 房屋优势（地铁）
def parse_advantage_subway(text):
    if not isinstance(text, str) or text == 'Unknown':
        return 0
    return 1 if '地铁' in text else 0

X_train['Advantage_Subway'] = X_train['房屋优势'].apply(parse_advantage_subway)
X_val['Advantage_Subway'] = X_val['房屋优势'].apply(parse_advantage_subway)
X_test_price['Advantage_Subway'] = X_test_price['房屋优势'].apply(parse_advantage_subway)

# 配备电梯
def parse_elevator(text):
    if not isinstance(text, str) or text == 'Unknown':
        return np.nan
    text_lower = text.lower()
    if '有' in text_lower:
        return 1
    elif '无' in text_lower:
        return 0
    else:
        return np.nan

X_train['Has_Elevator'] = X_train['配备电梯'].apply(parse_elevator)
X_val['Has_Elevator'] = X_val['配备电梯'].apply(parse_elevator)
X_test_price['Has_Elevator'] = X_test_price['配备电梯'].apply(parse_elevator)
elevator_mode = X_train['Has_Elevator'].mode()[0]
for df in [X_train, X_val, X_test_price]:
    df['Has_Elevator'] = df['Has_Elevator'].fillna(elevator_mode)

print("创建新特征: Property_Rights_Shared, Advantage_Subway, Has_Elevator")


创建新特征: Property_Rights_Shared, Advantage_Subway, Has_Elevator


In [21]:

# 目标编码准备
print("\n--- 步骤9: 目标编码 ---")

quantified_group1 = ['交通出行', '周边配套', '核心卖点'] 
quantified_group2 = ['建筑年代', '房屋总数', '物 业 费', '交易时间', '上次交易']
decided_to_drop = ['开发商', '物业公司']
advanced_quantified = ['房屋户型', '所在楼层', '梯户比例', '绿 化 率', '燃气费', '停车费用', '楼栋总数']
newly_processed = ['产权所属', '房屋优势', '配备电梯']

all_object_cols_to_drop = quantified_group1 + quantified_group2 + decided_to_drop + advanced_quantified + newly_processed
cols_to_drop_final_step = [col for col in all_object_cols_to_drop if col in X_train.columns]

X_train_pre_TE = X_train.drop(columns=cols_to_drop_final_step, errors='ignore').copy()
X_val_pre_TE = X_val.drop(columns=cols_to_drop_final_step, errors='ignore').copy()
X_test_price_pre_TE = X_test_price.drop(columns=cols_to_drop_final_step, errors='ignore').copy()

X_train_temp = X_train_pre_TE.copy()
X_train_temp['target_log'] = y_train_log

global_mean_log_price = y_train_log.mean()

# 目标编码函数
def target_encode_column_fixed(train_df, val_df, test_df, col, target_col_name, global_mean, alpha=5):
    category_means = train_df.groupby(col)[target_col_name].mean()
    category_counts = train_df.groupby(col)[target_col_name].count()
    encoded_values = (category_means * category_counts + global_mean * alpha) / (category_counts + alpha)
    encoding_map = encoded_values.to_dict()
    
    train_encoded = train_df[col].map(encoding_map).fillna(global_mean)
    val_encoded = val_df[col].map(encoding_map).fillna(global_mean)
    test_encoded = test_df[col].map(encoding_map).fillna(global_mean)
    
    return train_encoded, val_encoded, test_encoded, encoding_map

# 执行目标编码
cols_to_encode = X_train_pre_TE.select_dtypes(include=['object']).columns.tolist()
encoding_maps = {}

for col in cols_to_encode:
    train_encoded, val_encoded, test_encoded, encoding_map = target_encode_column_fixed(
        X_train_temp, X_val_pre_TE, X_test_price_pre_TE,
        col, 'target_log', global_mean_log_price, alpha=5
    )
    encoding_maps[col] = encoding_map
    new_col_name = f'{col}_TE'
    X_train_pre_TE[new_col_name] = train_encoded
    X_val_pre_TE[new_col_name] = val_encoded
    X_test_price_pre_TE[new_col_name] = test_encoded

X_train_post_TE = X_train_pre_TE.drop(columns=cols_to_encode)
X_val_post_TE = X_val_pre_TE.drop(columns=cols_to_encode)
X_test_price_post_TE = X_test_price_pre_TE.drop(columns=cols_to_encode)

print(f"目标编码完成，处理特征: {len(cols_to_encode)}个")



--- 步骤9: 目标编码 ---
目标编码完成，处理特征: 12个


In [22]:

# 城市、区域、板块目标编码
print("\n--- 步骤10: 地理特征目标编码 ---")
train_data_for_encoding = pd.concat([X_train_post_TE, y_train_log], axis=1)
target_col_name = y_train_log.name if y_train_log.name is not None else 'Price'
m = 20

# 城市编码
city_stats = train_data_for_encoding.groupby('城市')[target_col_name].agg(['mean', 'count'])
city_stats['TE_City_Smoothed'] = (city_stats['count'] * city_stats['mean'] + m * global_mean_log_price) / (city_stats['count'] + m)
city_encoding_map = city_stats['TE_City_Smoothed'].to_dict()
X_train_post_TE['TE_City'] = X_train_post_TE['城市'].map(city_encoding_map).fillna(global_mean_log_price)
X_val_post_TE['TE_City'] = X_val_post_TE['城市'].map(city_encoding_map).fillna(global_mean_log_price)
X_test_price_post_TE['TE_City'] = X_test_price_post_TE['城市'].map(city_encoding_map).fillna(global_mean_log_price)

# 区域编码
district_stats = train_data_for_encoding.groupby('区域')[target_col_name].agg(['mean', 'count'])
district_stats['TE_District_Smoothed'] = (district_stats['count'] * district_stats['mean'] + m * global_mean_log_price) / (district_stats['count'] + m)
district_encoding_map = district_stats['TE_District_Smoothed'].to_dict()
X_train_post_TE['TE_District'] = X_train_post_TE['区域'].map(district_encoding_map).fillna(global_mean_log_price)
X_val_post_TE['TE_District'] = X_val_post_TE['区域'].map(district_encoding_map).fillna(global_mean_log_price)
X_test_price_post_TE['TE_District'] = X_test_price_post_TE['区域'].map(district_encoding_map).fillna(global_mean_log_price)

# 板块编码
block_stats = train_data_for_encoding.groupby(['区域', '板块'])[target_col_name].agg(['mean', 'count'])
district_mean_map = district_stats['mean'].to_dict()

def stratified_smooth(row, district_means, global_m, global_mean):
    block_mean = row['mean']
    block_count = row['count']
    district_id = row.name[0] 
    district_mean = district_means.get(district_id, global_mean) 
    return (block_count * block_mean + global_m * district_mean) / (block_count + global_m)

block_stats['TE_Block_Stratified'] = block_stats.apply(stratified_smooth, axis=1, district_means=district_mean_map, global_m=m, global_mean=global_mean_log_price)
block_encoding_map = block_stats['TE_Block_Stratified'].to_dict()

def apply_block_encoding(row, block_map, district_map_smoothed, global_mean_val):
    district_id = row['区域']
    block_id = row['板块']
    key = (district_id, block_id)
    block_value = block_map.get(key)
    if block_value is not None:
        return block_value
    else:
        district_value = district_map_smoothed.get(district_id)
        return district_value if district_value is not None else global_mean_val

X_train_post_TE['TE_Block_Stratified'] = X_train_post_TE.apply(apply_block_encoding, axis=1, block_map=block_encoding_map, district_map_smoothed=district_encoding_map, global_mean_val=global_mean_log_price)
X_val_post_TE['TE_Block_Stratified'] = X_val_post_TE.apply(apply_block_encoding, axis=1, block_map=block_encoding_map, district_map_smoothed=district_encoding_map, global_mean_val=global_mean_log_price)
X_test_price_post_TE['TE_Block_Stratified'] = X_test_price_post_TE.apply(apply_block_encoding, axis=1, block_map=block_encoding_map, district_map_smoothed=district_encoding_map, global_mean_val=global_mean_log_price)



--- 步骤10: 地理特征目标编码 ---


In [23]:

# 清理原始ID列
original_id_cols_to_drop = ['城市', '区域', '板块']
cols_present_train = [col for col in original_id_cols_to_drop if col in X_train_post_TE.columns]
if cols_present_train:
    X_train_post_TE_cleaned = X_train_post_TE.drop(columns=cols_present_train)
    X_val_post_TE_cleaned = X_val_post_TE.drop(columns=cols_present_train)
    X_test_price_post_TE_cleaned = X_test_price_post_TE.drop(columns=cols_present_train)
else:
    X_train_post_TE_cleaned = X_train_post_TE.copy()
    X_val_post_TE_cleaned = X_val_post_TE.copy()
    X_test_price_post_TE_cleaned = X_test_price_post_TE.copy()

print("地理特征目标编码完成: TE_City, TE_District, TE_Block_Stratified")


地理特征目标编码完成: TE_City, TE_District, TE_Block_Stratified


In [24]:

# One-Hot编码
print("\n--- 步骤11: One-Hot编码 ---")
categorical_cols_remaining_for_ohe = X_train_post_TE_cleaned.select_dtypes(include=['object', 'category', 'string']).columns.tolist()

if categorical_cols_remaining_for_ohe:
    X_train_final = pd.get_dummies(X_train_post_TE_cleaned, columns=categorical_cols_remaining_for_ohe, dummy_na=False, dtype=int, drop_first=True)
    X_val_final = pd.get_dummies(X_val_post_TE_cleaned, columns=categorical_cols_remaining_for_ohe, dummy_na=False, dtype=int, drop_first=True)
    X_test_price_final = pd.get_dummies(X_test_price_post_TE_cleaned, columns=categorical_cols_remaining_for_ohe, dummy_na=False, dtype=int, drop_first=True)
    
    final_train_columns = X_train_final.columns
    X_val_final = X_val_final.reindex(columns=final_train_columns, fill_value=0)
    X_test_price_final = X_test_price_final.reindex(columns=final_train_columns, fill_value=0)
    print(f"OHE编码完成，处理特征: {len(categorical_cols_remaining_for_ohe)}个")
else:
    X_train_final = X_train_post_TE_cleaned.copy()
    X_val_final = X_val_post_TE_cleaned.copy()
    X_test_price_final = X_test_price_post_TE_cleaned.copy()
    print("无需OHE编码")



--- 步骤11: One-Hot编码 ---
无需OHE编码


In [25]:

# 最终清理
print("\n--- 步骤12: 最终清理 ---")
final_cols_to_drop = [
    '套内面积', '年份', '区县', '板块_comm', 
    'coord_x', 'coord_y', 'Total_Floors', 
    'Num_Elevators', 'Num_Households'
]

cols_present_train = [col for col in final_cols_to_drop if col in X_train_final.columns]
if cols_present_train:
    X_train_final_cleaned = X_train_final.drop(columns=cols_present_train)
    X_val_final_cleaned = X_val_final.drop(columns=cols_present_train)
    X_test_price_final_cleaned = X_test_price_final.drop(columns=cols_present_train)
else:
    X_train_final_cleaned = X_train_final.copy()
    X_val_final_cleaned = X_val_final.copy()
    X_test_price_final_cleaned = X_test_price_final.copy()

print("最终清理完成")



--- 步骤12: 最终清理 ---
最终清理完成


In [26]:

# Winsorization（缩尾处理）
print("\n--- 步骤13: 缩尾处理 ---")
original_numeric_cols_to_keep = [
    '建筑面积', 'lon', 'lat', '容 积 率', '停车位', 
    'Has_交通出行', 'Has_周边配套', 'Has_核心卖点', 
    'House_Age', 'Property_Fee', 'Total_Units', 
    'Transaction_Year', 'Transaction_Month', 'Years_Since_Last_Transaction', 
    'Num_Rooms', 'Num_LivingRooms', 'Num_Kitchens', 'Num_Baths', 
    'Floor_Ratio', 'Estimated_Floor', 'Elevator_Ratio', 
    'Greening_Rate', 'Gas_Fee', 'Parking_Fee', 
    'Total_Buildings', 'Area_sq',
    'Property_Rights_Shared', 'Advantage_Subway', 'Has_Elevator'
]

TE_cols = ['TE_City', 'TE_District', 'TE_Block_Stratified']
cols_to_winsorize_final = original_numeric_cols_to_keep + TE_cols
cols_to_winsorize_present = [col for col in cols_to_winsorize_final if col in X_train_final_cleaned.columns]

if cols_to_winsorize_present:
    bounds = {}
    lower_quantile, upper_quantile = 0.01, 0.99
    
    for col in cols_to_winsorize_present:
        q_lower = X_train_final_cleaned[col].quantile(lower_quantile)
        q_upper = X_train_final_cleaned[col].quantile(upper_quantile)
        if q_lower < q_upper:
            bounds[col] = (q_lower, q_upper)
    
    X_train_winsorized = X_train_final_cleaned.copy()
    X_val_winsorized = X_val_final_cleaned.copy()
    X_test_price_winsorized = X_test_price_final_cleaned.copy()
    
    for col, (q_lower, q_upper) in bounds.items():
        X_train_winsorized[col] = X_train_winsorized[col].clip(lower=q_lower, upper=q_upper)
        X_val_winsorized[col] = X_val_winsorized[col].clip(lower=q_lower, upper=q_upper)
        X_test_price_winsorized[col] = X_test_price_winsorized[col].clip(lower=q_lower, upper=q_upper)
    
    X_train_final_cleaned = X_train_winsorized
    X_val_final_cleaned = X_val_winsorized
    X_test_price_final_cleaned = X_test_price_winsorized
    
    print(f"缩尾处理完成，处理特征: {len(bounds)}个")



--- 步骤13: 缩尾处理 ---
缩尾处理完成，处理特征: 31个


In [27]:

# 最终汇总
print("\n" + "="*70)
print("特征工程全部完成！")
print("="*70)

print(f"最终数据维度:")
print(f"  训练集: {X_train_final_cleaned.shape}")
print(f"  验证集: {X_val_final_cleaned.shape}")
print(f"  测试集: {X_test_price_final_cleaned.shape}")
print(f"  目标变量: {y_train_log.shape} (对数转换后)")

print(f"\n最终特征数量: {X_train_final_cleaned.shape[1]}")

te_cols = [col for col in X_train_final_cleaned.columns if col.startswith('TE_')]
print(f"目标编码特征: {len(te_cols)}个")

print("\n下一步:")
print("  1. 模型训练 (使用 X_train_final_cleaned, y_train_log)")
print("  2. 模型验证 (使用 X_val_final_cleaned, y_val_log)") 
print("  3. 最终预测 (使用 X_test_price_final_cleaned)")
print("重要提醒: 预测完成后使用 np.expm1() 将结果转换回原始价格尺度")


特征工程全部完成！
最终数据维度:
  训练集: (83096, 44)
  验证集: (20775, 44)
  测试集: (34017, 44)
  目标变量: (83096,) (对数转换后)

最终特征数量: 44
目标编码特征: 3个

下一步:
  1. 模型训练 (使用 X_train_final_cleaned, y_train_log)
  2. 模型验证 (使用 X_val_final_cleaned, y_val_log)
  3. 最终预测 (使用 X_test_price_final_cleaned)
重要提醒: 预测完成后使用 np.expm1() 将结果转换回原始价格尺度


开始建立模型

In [28]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import LinearRegression, Lasso, Ridge, ElasticNet
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

print("\n" + "="*80)
print("开始模型训练和优化...")
print("="*80)

# 数据标准化
scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train_final_cleaned)
X_val_scaled = scaler.transform(X_val_final_cleaned)
X_test_scaled = scaler.transform(X_test_price_final_cleaned)

print(f"✓ 数据标准化完成")



开始模型训练和优化...
✓ 数据标准化完成


In [29]:

# 定义评估函数
def evaluate_model(model, X_train, y_train, X_val, y_val, model_name):
    """评估模型性能"""
    
    # 训练集预测
    y_train_pred = model.predict(X_train)
    train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
    train_r2 = r2_score(y_train, y_train_pred)
    
    # 验证集预测
    y_val_pred = model.predict(X_val)
    val_rmse = np.sqrt(mean_squared_error(y_val, y_val_pred))
    val_r2 = r2_score(y_val, y_val_pred)
    val_mae = mean_absolute_error(y_val, y_val_pred)
    
    results = {
        'model_name': model_name,
        'train_rmse': train_rmse,
        'val_rmse': val_rmse,
        'train_r2': train_r2,
        'val_r2': val_r2,
        'val_mae': val_mae
    }
    
    return results, y_val_pred


In [ ]:

# 模型训练和比较
models_results = []

print("\n--- 线性模型 ---")

# 1. 线性回归
print("训练线性回归...")
linear_model = LinearRegression()
linear_model.fit(X_train_scaled, y_train_log)
linear_results, linear_pred = evaluate_model(linear_model, X_train_scaled, y_train_log, X_val_scaled, y_val_log, "Linear Regression")
models_results.append(linear_results)
print(f"  RMSE: {linear_results['val_rmse']:.4f}, R²: {linear_results['val_r2']:.4f}")

# 输出线性回归预测结果
linear_test_pred_log = linear_model.predict(X_test_scaled)
linear_test_pred = np.expm1(linear_test_pred_log)
linear_submission = pd.DataFrame({
    'ID': range(len(linear_test_pred)),
    'Price': linear_test_pred
})
linear_submission.to_csv('linear_regression_predictions.csv', index=False)
print(f"✓ 线性回归预测结果已保存到 linear_regression_predictions.csv")



--- 线性模型 ---
训练线性回归...
  RMSE: 0.2624, R²: 0.8983
✓ 线性回归预测结果已保存到 linear_regression_predictions.csv


In [31]:

# 2. Lasso回归
print("训练Lasso回归...")
lasso_params = {'alpha': [0.001, 0.01, 0.1, 1, 10]}
lasso_grid = GridSearchCV(Lasso(random_state=111, max_iter=1000), lasso_params, 
                          cv=6, scoring='neg_mean_squared_error', n_jobs=-1)  # 改为6折
lasso_grid.fit(X_train_scaled, y_train_log)
lasso_model = lasso_grid.best_estimator_
lasso_results, lasso_pred = evaluate_model(lasso_model, X_train_scaled, y_train_log, X_val_scaled, y_val_log, "Lasso")
models_results.append(lasso_results)
print(f"  最佳alpha: {lasso_grid.best_params_['alpha']}")
print(f"  RMSE: {lasso_results['val_rmse']:.4f}, R²: {lasso_results['val_r2']:.4f}")

# 输出Lasso回归预测结果
lasso_test_pred_log = lasso_model.predict(X_test_scaled)
lasso_test_pred = np.expm1(lasso_test_pred_log)
lasso_submission = pd.DataFrame({
    'ID': range(len(lasso_test_pred)),
    'Price': lasso_test_pred
})
lasso_submission.to_csv('lasso_predictions.csv', index=False)
print(f"✓ Lasso回归预测结果已保存到 lasso_predictions.csv")


训练Lasso回归...
  最佳alpha: 0.001
  RMSE: 0.2633, R²: 0.8977
✓ Lasso回归预测结果已保存到 lasso_predictions.csv


In [32]:

# 3. 岭回归
print("训练岭回归...")
ridge_params = {'alpha': [0.001, 0.01, 0.1, 1, 10, 100]}
ridge_grid = GridSearchCV(Ridge(random_state=111), ridge_params, 
                         cv=6, scoring='neg_mean_squared_error', n_jobs=-1)  # 改为6折
ridge_grid.fit(X_train_scaled, y_train_log)
ridge_model = ridge_grid.best_estimator_
ridge_results, ridge_pred = evaluate_model(ridge_model, X_train_scaled, y_train_log, X_val_scaled, y_val_log, "Ridge")
models_results.append(ridge_results)
print(f"  最佳alpha: {ridge_grid.best_params_['alpha']}")
print(f"  RMSE: {ridge_results['val_rmse']:.4f}, R²: {ridge_results['val_r2']:.4f}")

# 输出岭回归预测结果
ridge_test_pred_log = ridge_model.predict(X_test_scaled)
ridge_test_pred = np.expm1(ridge_test_pred_log)
ridge_submission = pd.DataFrame({
    'ID': range(len(ridge_test_pred)),
    'Price': ridge_test_pred
})
ridge_submission.to_csv('ridge_predictions.csv', index=False)
print(f"✓ 岭回归预测结果已保存到 ridge_predictions.csv")


训练岭回归...
  最佳alpha: 1
  RMSE: 0.2624, R²: 0.8983
✓ 岭回归预测结果已保存到 ridge_predictions.csv


In [33]:

# 4. 弹性网络
print("训练弹性网络...")
elastic_params = {
    'alpha': [0.001, 0.01, 0.1, 1, 10],
    'l1_ratio': [0.1, 0.3, 0.5, 0.7, 0.9]
}
elastic_grid = GridSearchCV(ElasticNet(random_state=111, max_iter=1000), elastic_params, 
                           cv=6, scoring='neg_mean_squared_error', n_jobs=-1)  # 改为6折
elastic_grid.fit(X_train_scaled, y_train_log)
elastic_model = elastic_grid.best_estimator_
elastic_results, elastic_pred = evaluate_model(elastic_model, X_train_scaled, y_train_log, X_val_scaled, y_val_log, "Elastic Net")
models_results.append(elastic_results)
print(f"  最佳参数: alpha={elastic_grid.best_params_['alpha']}, l1_ratio={elastic_grid.best_params_['l1_ratio']}")
print(f"  RMSE: {elastic_results['val_rmse']:.4f}, R²: {elastic_results['val_r2']:.4f}")

# 输出弹性网络预测结果
elastic_test_pred_log = elastic_model.predict(X_test_scaled)
elastic_test_pred = np.expm1(elastic_test_pred_log)
elastic_submission = pd.DataFrame({
    'ID': range(len(elastic_test_pred)),
    'Price': elastic_test_pred
})
elastic_submission.to_csv('elastic_net_predictions.csv', index=False)
print(f"✓ 弹性网络预测结果已保存到 elastic_net_predictions.csv")


训练弹性网络...
  最佳参数: alpha=0.001, l1_ratio=0.1
  RMSE: 0.2625, R²: 0.8983
✓ 弹性网络预测结果已保存到 elastic_net_predictions.csv


In [34]:

print("\n--- 模型比较和分析 ---")

# 创建结果比较表格
results_df = pd.DataFrame(models_results)
results_df = results_df.sort_values('val_rmse')

print("\n" + "="*80)
print("模型性能比较")
print("="*80)
print(results_df.round(4))

# 选择最佳模型
best_model_info = results_df.iloc[0]
best_model_name = best_model_info['model_name']
best_val_rmse = best_model_info['val_rmse']

print(f"\n最佳模型: {best_model_name}")
print(f"最佳验证集RMSE: {best_val_rmse:.4f}")

print("\n" + "="*80)
print("模型训练总结")
print("="*80)
print(f"1. 数据集: 训练集 {X_train_final_cleaned.shape}, 验证集 {X_val_final_cleaned.shape}")
print(f"2. 特征数量: {X_train_final_cleaned.shape[1]}")
print(f"3. 模型比较: 测试了 {len(models_results)} 个模型")
print(f"4. 最佳模型: {best_model_name}")
print(f"5. 最佳RMSE: {best_val_rmse:.4f} (对数尺度)")
print(f"6. 预测结果: 已生成4个CSV文件")




--- 模型比较和分析 ---

模型性能比较
          model_name  train_rmse  val_rmse  train_r2  val_r2  val_mae
2              Ridge      0.2633    0.2624    0.8998  0.8983   0.1988
0  Linear Regression      0.2633    0.2624    0.8998  0.8983   0.1988
3        Elastic Net      0.2634    0.2625    0.8997  0.8983   0.1990
1              Lasso      0.2639    0.2633    0.8993  0.8977   0.1995

最佳模型: Ridge
最佳验证集RMSE: 0.2624

模型训练总结
1. 数据集: 训练集 (83096, 44), 验证集 (20775, 44)
2. 特征数量: 44
3. 模型比较: 测试了 4 个模型
4. 最佳模型: Ridge
5. 最佳RMSE: 0.2624 (对数尺度)
6. 预测结果: 已生成4个CSV文件


下面代码块与上一个代码块相比仅添加交互项，其余操作完全相同

In [39]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import LinearRegression, Lasso, Ridge, ElasticNet
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

print("\n" + "="*80)
print("开始模型训练和优化（含交互项对比）...")
print("="*80)

# 首先备份原始数据
X_train_original = X_train_final_cleaned.copy()
X_val_original = X_val_final_cleaned.copy()
X_test_original = X_test_price_final_cleaned.copy()

print(f"原始特征数量: {X_train_original.shape[1]}")

# 添加交互项
print("\n--- 添加交互项 ---")

def add_interaction_features(X_train, X_val, X_test):
    """添加业务逻辑驱动的交互项"""
    
    # 创建副本
    X_train_int = X_train.copy()
    X_val_int = X_val.copy()
    X_test_int = X_test.copy()
    
    # 1. 区域价值与物业品质的交互
    if all(col in X_train.columns for col in ['TE_District', 'Property_Fee']):
        X_train_int['TE_District_x_Property_Fee'] = X_train_int['TE_District'] * X_train_int['Property_Fee']
        X_val_int['TE_District_x_Property_Fee'] = X_val_int['TE_District'] * X_val_int['Property_Fee']
        X_test_int['TE_District_x_Property_Fee'] = X_test_int['TE_District'] * X_test_int['Property_Fee']
        print("✓ 添加: 区域价值 × 物业品质")
    
    # 2. 单位面积价值指标
    if all(col in X_train.columns for col in ['TE_Block_Stratified', '建筑面积']):
        X_train_int['Price_Per_Sqm_Estimate'] = X_train_int['TE_Block_Stratified'] / (X_train_int['建筑面积'] + 1)
        X_val_int['Price_Per_Sqm_Estimate'] = X_val_int['TE_Block_Stratified'] / (X_val_int['建筑面积'] + 1)
        X_test_int['Price_Per_Sqm_Estimate'] = X_test_int['TE_Block_Stratified'] / (X_test_int['建筑面积'] + 1)
        print("✓ 添加: 单位面积价值指标")
    
    # 3. 相对楼层位置
    if all(col in X_train.columns for col in ['Estimated_Floor', 'Total_Floors']):
        X_train_int['Floor_Position_Ratio'] = X_train_int['Estimated_Floor'] / (X_train_int['Total_Floors'] + 1)
        X_val_int['Floor_Position_Ratio'] = X_val_int['Estimated_Floor'] / (X_val_int['Total_Floors'] + 1)
        X_test_int['Floor_Position_Ratio'] = X_test_int['Estimated_Floor'] / (X_test_int['Total_Floors'] + 1)
        print("✓ 添加: 相对楼层位置")
    
    # 4. 交通与配套组合优势
    if all(col in X_train.columns for col in ['Has_交通出行', 'Has_周边配套']):
        X_train_int['Transport_Plus_Amenities'] = X_train_int['Has_交通出行'] * X_train_int['Has_周边配套']
        X_val_int['Transport_Plus_Amenities'] = X_val_int['Has_交通出行'] * X_val_int['Has_周边配套']
        X_test_int['Transport_Plus_Amenities'] = X_test_int['Has_交通出行'] * X_test_int['Has_周边配套']
        print("✓ 添加: 交通+配套组合优势")
    
    # 5. 老房有电梯的特殊价值
    if all(col in X_train.columns for col in ['House_Age', 'Has_Elevator']):
        X_train_int['Old_Building_With_Elevator'] = (X_train_int['House_Age'] > 20) * X_train_int['Has_Elevator']
        X_val_int['Old_Building_With_Elevator'] = (X_val_int['House_Age'] > 20) * X_val_int['Has_Elevator']
        X_test_int['Old_Building_With_Elevator'] = (X_test_int['House_Age'] > 20) * X_test_int['Has_Elevator']
        print("✓ 添加: 老房有电梯特殊价值")
    
    # 6. 建筑密度与绿化的平衡
    if all(col in X_train.columns for col in ['容 积 率', 'Greening_Rate']):
        X_train_int['Density_Green_Balance'] = X_train_int['容 积 率'] * X_train_int['Greening_Rate']
        X_val_int['Density_Green_Balance'] = X_val_int['容 积 率'] * X_val_int['Greening_Rate']
        X_test_int['Density_Green_Balance'] = X_test_int['容 积 率'] * X_test_int['Greening_Rate']
        print("✓ 添加: 建筑密度与绿化平衡")
    
    print(f"交互项添加完成，新特征数量: {X_train_int.shape[1]}")
    return X_train_int, X_val_int, X_test_int

# 添加交互项
X_train_with_interactions, X_val_with_interactions, X_test_with_interactions = add_interaction_features(
    X_train_original, X_val_original, X_test_original
)

# 定义评估函数
def evaluate_model(model, X_train, y_train, X_val, y_val, model_name):
    """评估模型性能"""
    
    # 训练集预测
    y_train_pred = model.predict(X_train)
    train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
    train_r2 = r2_score(y_train, y_train_pred)
    
    # 验证集预测
    y_val_pred = model.predict(X_val)
    val_rmse = np.sqrt(mean_squared_error(y_val, y_val_pred))
    val_r2 = r2_score(y_val, y_val_pred)
    val_mae = mean_absolute_error(y_val, y_val_pred)
    
    results = {
        'model_name': model_name,
        'train_rmse': train_rmse,
        'val_rmse': val_rmse,
        'train_r2': train_r2,
        'val_r2': val_r2,
        'val_mae': val_mae
    }
    
    return results, y_val_pred

# 训练函数
def train_and_evaluate_models(X_train, X_val, X_test, feature_type):
    """训练并评估所有模型"""
    models_results = []
    trained_models = {}
    
    print(f"\n--- {feature_type}特征 - 线性模型 ---")
    
    # 数据标准化
    scaler = RobustScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    X_test_scaled = scaler.transform(X_test)
    
    # 1. 线性回归
    print("训练线性回归...")
    linear_model = LinearRegression()
    linear_model.fit(X_train_scaled, y_train_log)
    linear_results, linear_pred = evaluate_model(linear_model, X_train_scaled, y_train_log, X_val_scaled, y_val_log, "Linear Regression")
    models_results.append(linear_results)
    trained_models['linear'] = linear_model
    
    # 输出线性回归预测结果
    linear_test_pred_log = linear_model.predict(X_test_scaled)
    linear_test_pred = np.expm1(linear_test_pred_log)
    linear_submission = pd.DataFrame({
        'ID': range(len(linear_test_pred)),
        'Price': linear_test_pred
    })
    linear_submission.to_csv(f'{feature_type}_linear_regression_predictions.csv', index=False)
    print(f"  RMSE: {linear_results['val_rmse']:.4f}, R²: {linear_results['val_r2']:.4f}")
    print(f"  ✓ {feature_type}线性回归预测结果已保存")

    # 2. Lasso回归
    print("训练Lasso回归...")
    lasso_params = {'alpha': [0.001, 0.01, 0.1, 1, 10]}
    lasso_grid = GridSearchCV(Lasso(random_state=111, max_iter=1000), lasso_params, 
                              cv=6, scoring='neg_mean_squared_error', n_jobs=-1)  # 改为6折
    lasso_grid.fit(X_train_scaled, y_train_log)
    lasso_model = lasso_grid.best_estimator_
    lasso_results, lasso_pred = evaluate_model(lasso_model, X_train_scaled, y_train_log, X_val_scaled, y_val_log, "Lasso")
    models_results.append(lasso_results)
    trained_models['lasso'] = lasso_model
    
    # 输出Lasso回归预测结果
    lasso_test_pred_log = lasso_model.predict(X_test_scaled)
    lasso_test_pred = np.expm1(lasso_test_pred_log)
    lasso_submission = pd.DataFrame({
        'ID': range(len(lasso_test_pred)),
        'Price': lasso_test_pred
    })
    lasso_submission.to_csv(f'{feature_type}_lasso_predictions.csv', index=False)
    print(f"  最佳alpha: {lasso_grid.best_params_['alpha']}")
    print(f"  RMSE: {lasso_results['val_rmse']:.4f}, R²: {lasso_results['val_r2']:.4f}")
    print(f"  ✓ {feature_type}Lasso回归预测结果已保存")

    # 3. 岭回归
    print("训练岭回归...")
    ridge_params = {'alpha': [0.001, 0.01, 0.1, 1, 10, 100]}
    ridge_grid = GridSearchCV(Ridge(random_state=111), ridge_params, 
                             cv=6, scoring='neg_mean_squared_error', n_jobs=-1)  # 改为6折
    ridge_grid.fit(X_train_scaled, y_train_log)
    ridge_model = ridge_grid.best_estimator_
    ridge_results, ridge_pred = evaluate_model(ridge_model, X_train_scaled, y_train_log, X_val_scaled, y_val_log, "Ridge")
    models_results.append(ridge_results)
    trained_models['ridge'] = ridge_model
    
    # 输出岭回归预测结果
    ridge_test_pred_log = ridge_model.predict(X_test_scaled)
    ridge_test_pred = np.expm1(ridge_test_pred_log)
    ridge_submission = pd.DataFrame({
        'ID': range(len(ridge_test_pred)),
        'Price': ridge_test_pred
    })
    ridge_submission.to_csv(f'{feature_type}_ridge_predictions.csv', index=False)
    print(f"  最佳alpha: {ridge_grid.best_params_['alpha']}")
    print(f"  RMSE: {ridge_results['val_rmse']:.4f}, R²: {ridge_results['val_r2']:.4f}")
    print(f"  ✓ {feature_type}岭回归预测结果已保存")

    # 4. 弹性网络
    print("训练弹性网络...")
    elastic_params = {
        'alpha': [0.001, 0.01, 0.1, 1, 10],
        'l1_ratio': [0.1, 0.3, 0.5, 0.7, 0.9]
    }
    elastic_grid = GridSearchCV(ElasticNet(random_state=111, max_iter=1000), elastic_params, 
                               cv=6, scoring='neg_mean_squared_error', n_jobs=-1)  # 改为6折
    elastic_grid.fit(X_train_scaled, y_train_log)
    elastic_model = elastic_grid.best_estimator_
    elastic_results, elastic_pred = evaluate_model(elastic_model, X_train_scaled, y_train_log, X_val_scaled, y_val_log, "Elastic Net")
    models_results.append(elastic_results)
    trained_models['elastic'] = elastic_model
    
    # 输出弹性网络预测结果
    elastic_test_pred_log = elastic_model.predict(X_test_scaled)
    elastic_test_pred = np.expm1(elastic_test_pred_log)
    elastic_submission = pd.DataFrame({
        'ID': range(len(elastic_test_pred)),
        'Price': elastic_test_pred
    })
    elastic_submission.to_csv(f'{feature_type}_elastic_net_predictions.csv', index=False)
    print(f"  最佳参数: alpha={elastic_grid.best_params_['alpha']}, l1_ratio={elastic_grid.best_params_['l1_ratio']}")
    print(f"  RMSE: {elastic_results['val_rmse']:.4f}, R²: {elastic_results['val_r2']:.4f}")
    print(f"  ✓ {feature_type}弹性网络预测结果已保存")
    
    return models_results, trained_models

# 分别用原始特征和交互项特征训练模型
print("\n" + "="*80)
print("原始特征模型训练")
print("="*80)
original_results, original_models = train_and_evaluate_models(
    X_train_original, X_val_original, X_test_original, "原始"
)

print("\n" + "="*80)
print("交互项特征模型训练")
print("="*80)
interaction_results, interaction_models = train_and_evaluate_models(
    X_train_with_interactions, X_val_with_interactions, X_test_with_interactions, "交互项"
)

# 性能对比分析
print("\n" + "="*80)
print("性能对比分析")
print("="*80)

# 创建对比表格
original_df = pd.DataFrame(original_results)
interaction_df = pd.DataFrame(interaction_results)

# 合并对比结果
comparison_df = pd.merge(
    original_df, 
    interaction_df, 
    on='model_name', 
    suffixes=('_原始', '_交互项')
)

# 计算性能提升
comparison_df['RMSE_提升'] = comparison_df['val_rmse_原始'] - comparison_df['val_rmse_交互项']
comparison_df['R2_提升'] = comparison_df['val_r2_交互项'] - comparison_df['val_r2_原始']

print("\n模型性能对比:")
print(comparison_df[['model_name', 'val_rmse_原始', 'val_rmse_交互项', 'RMSE_提升', 
                     'val_r2_原始', 'val_r2_交互项', 'R2_提升']].round(4))

# 找到最佳模型（交互项特征）
interaction_df_sorted = interaction_df.sort_values('val_rmse')
best_interaction_model_info = interaction_df_sorted.iloc[0]
best_interaction_model_name = best_interaction_model_info['model_name']

# 创建模型名称到字典键的映射
model_name_to_key = {
    'Linear Regression': 'linear',
    'Lasso': 'lasso', 
    'Ridge': 'ridge',
    'Elastic Net': 'elastic'
}

# 获取对应的字典键
model_key = model_name_to_key.get(best_interaction_model_name)

if model_key is None:
    print(f"警告: 找不到模型 {best_interaction_model_name} 对应的键，使用默认的第一个模型")
    # 使用默认的第一个模型
    model_key = list(interaction_models.keys())[0]
    # 反向查找模型名称
    best_interaction_model_name = [k for k, v in model_name_to_key.items() if v == model_key][0]
    print(f"使用默认模型: {best_interaction_model_name}")

# 获取最佳模型
best_interaction_model = interaction_models[model_key]

print(f"\n🏆 交互项特征最佳模型: {best_interaction_model_name}")
print(f"   验证集RMSE: {best_interaction_model_info['val_rmse']:.4f}")
print(f"   验证集R²: {best_interaction_model_info['val_r2']:.4f}")

print("\n" + "="*80)
print("交互项特征分析总结")
print("="*80)
print(f"1. 原始特征数量: {X_train_original.shape[1]}")
print(f"2. 交互项特征数量: {X_train_with_interactions.shape[1]}")
print(f"3. 新增交互项: {X_train_with_interactions.shape[1] - X_train_original.shape[1]}个")

# 统计性能提升
improved_models = comparison_df[comparison_df['RMSE_提升'] > 0]
if len(improved_models) > 0:
    avg_rmse_improvement = improved_models['RMSE_提升'].mean()
    best_improvement = improved_models['RMSE_提升'].max()
    print(f"4. 性能提升统计:")
    print(f"   - 改进模型数量: {len(improved_models)}/{len(comparison_df)}")
    print(f"   - 平均RMSE提升: {avg_rmse_improvement:.4f}")
    print(f"   - 最大RMSE提升: {best_improvement:.4f}")
else:
    print("4. 性能提升: 无显著改进")

print(f"5. 最佳模型: {best_interaction_model_name}")
print(f"6. 最终RMSE: {best_interaction_model_info['val_rmse']:.4f}")

print("\n🔍 交互项特征效果分析:")
if len(improved_models) > len(comparison_df) * 0.5:
    print("  互项特征在多数模型中带来性能提升")
    print("  建议: 在后续建模中保留这些交互项")
else:
    print("  交互项特征效果有限")
    print("  建议: 可能需要调整交互项选择或尝试其他组合")

print("\n📁 生成的文件:")
print("  原始特征:")
print("    - 原始_linear_regression_predictions.csv")
print("    - 原始_lasso_predictions.csv") 
print("    - 原始_ridge_predictions.csv")
print("    - 原始_elastic_net_predictions.csv")
print("  交互项特征:")
print("    - 交互项_linear_regression_predictions.csv")
print("    - 交互项_lasso_predictions.csv")
print("    - 交互项_ridge_predictions.csv")
print("    - 交互项_elastic_net_predictions.csv")

print("\n✅ 交互项特征对比实验完成！")


开始模型训练和优化（含交互项对比）...
原始特征数量: 44

--- 添加交互项 ---
✓ 添加: 区域价值 × 物业品质
✓ 添加: 单位面积价值指标
✓ 添加: 交通+配套组合优势
✓ 添加: 老房有电梯特殊价值
✓ 添加: 建筑密度与绿化平衡
交互项添加完成，新特征数量: 49

原始特征模型训练

--- 原始特征 - 线性模型 ---
训练线性回归...
  RMSE: 0.2624, R²: 0.8983
  ✓ 原始线性回归预测结果已保存
训练Lasso回归...
  最佳alpha: 0.001
  RMSE: 0.2633, R²: 0.8977
  ✓ 原始Lasso回归预测结果已保存
训练岭回归...
  最佳alpha: 1
  RMSE: 0.2624, R²: 0.8983
  ✓ 原始岭回归预测结果已保存
训练弹性网络...
  最佳参数: alpha=0.001, l1_ratio=0.1
  RMSE: 0.2625, R²: 0.8983
  ✓ 原始弹性网络预测结果已保存

交互项特征模型训练

--- 交互项特征 - 线性模型 ---
训练线性回归...
  RMSE: 0.2584, R²: 0.9014
  ✓ 交互项线性回归预测结果已保存
训练Lasso回归...
  最佳alpha: 0.001
  RMSE: 0.2592, R²: 0.9008
  ✓ 交互项Lasso回归预测结果已保存
训练岭回归...
  最佳alpha: 10
  RMSE: 0.2584, R²: 0.9014
  ✓ 交互项岭回归预测结果已保存
训练弹性网络...
  最佳参数: alpha=0.001, l1_ratio=0.1
  RMSE: 0.2585, R²: 0.9013
  ✓ 交互项弹性网络预测结果已保存

性能对比分析

模型性能对比:
          model_name  val_rmse_原始  val_rmse_交互项  RMSE_提升  val_r2_原始  \
0  Linear Regression       0.2624        0.2584   0.0040     0.8983   
1              Lasso       0.2633        0.2592  

以下代码仅在原基础上添加交叉验证输出，无实际意义，由于再原基础上添加较为繁杂可直接查看输出表格

In [40]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import LinearRegression, Lasso, Ridge, ElasticNet
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

print("\n" + "="*80)
print("开始模型训练和优化...")
print("="*80)

# 数据标准化
scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train_final_cleaned)
X_val_scaled = scaler.transform(X_val_final_cleaned)
X_test_scaled = scaler.transform(X_test_price_final_cleaned)

print(f"✓ 数据标准化完成")

# 🔥 新增：定义交叉验证评估函数
def evaluate_model_with_cv(model, X_train, y_train, X_val, y_val, model_name, cv_folds=5):
    """评估模型性能，包括交叉验证"""
    
    # 训练集预测
    y_train_pred = model.predict(X_train)
    train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
    train_r2 = r2_score(y_train, y_train_pred)
    
    # 验证集预测
    y_val_pred = model.predict(X_val)
    val_rmse = np.sqrt(mean_squared_error(y_val, y_val_pred))
    val_r2 = r2_score(y_val, y_val_pred)
    val_mae = mean_absolute_error(y_val, y_val_pred)
    
    # 🔥 新增：交叉验证R²
    cv_scores = cross_val_score(model, X_train, y_train, cv=cv_folds, scoring='r2')
    cv_r2_mean = cv_scores.mean()
    cv_r2_std = cv_scores.std()
    
    results = {
        'model_name': model_name,
        'train_rmse': train_rmse,
        'val_rmse': val_rmse,
        'train_r2': train_r2,
        'val_r2': val_r2,
        'val_mae': val_mae,
        'cv_r2_mean': cv_r2_mean,  # 🔥 新增
        'cv_r2_std': cv_r2_std,    # 🔥 新增
        'cv_folds': cv_folds        # 🔥 新增
    }
    
    return results, y_val_pred, cv_scores

# 模型训练和比较
models_results = []

print("\n--- 线性模型 ---")

# 1. 线性回归
print("训练线性回归...")
linear_model = LinearRegression()
linear_model.fit(X_train_scaled, y_train_log)
linear_results, linear_pred, linear_cv_scores = evaluate_model_with_cv(
    linear_model, X_train_scaled, y_train_log, X_val_scaled, y_val_log, "Linear Regression"
)
models_results.append(linear_results)
print(f"  RMSE: {linear_results['val_rmse']:.4f}, R²: {linear_results['val_r2']:.4f}")
print(f"  交叉验证R²: {linear_results['cv_r2_mean']:.4f} ± {linear_results['cv_r2_std']:.4f}")

# 输出线性回归预测结果
linear_test_pred_log = linear_model.predict(X_test_scaled)
linear_test_pred = np.expm1(linear_test_pred_log)
linear_submission = pd.DataFrame({
    'ID': range(len(linear_test_pred)),
    'Price': linear_test_pred
})
linear_submission.to_csv('linear_regression_predictions.csv', index=False)
print(f"✓ 线性回归预测结果已保存到 linear_regression_predictions.csv")

# 2. Lasso回归
print("训练Lasso回归...")
lasso_params = {'alpha': [0.001, 0.01, 0.1, 1, 10]}
lasso_grid = GridSearchCV(Lasso(random_state=111, max_iter=1000), lasso_params, 
                          cv=6, scoring='neg_mean_squared_error', n_jobs=-1)
lasso_grid.fit(X_train_scaled, y_train_log)
lasso_model = lasso_grid.best_estimator_
lasso_results, lasso_pred, lasso_cv_scores = evaluate_model_with_cv(
    lasso_model, X_train_scaled, y_train_log, X_val_scaled, y_val_log, "Lasso"
)
models_results.append(lasso_results)
print(f"  最佳alpha: {lasso_grid.best_params_['alpha']}")
print(f"  RMSE: {lasso_results['val_rmse']:.4f}, R²: {lasso_results['val_r2']:.4f}")
print(f"  交叉验证R²: {lasso_results['cv_r2_mean']:.4f} ± {lasso_results['cv_r2_std']:.4f}")

# 输出Lasso回归预测结果
lasso_test_pred_log = lasso_model.predict(X_test_scaled)
lasso_test_pred = np.expm1(lasso_test_pred_log)
lasso_submission = pd.DataFrame({
    'ID': range(len(lasso_test_pred)),
    'Price': lasso_test_pred
})
lasso_submission.to_csv('lasso_predictions.csv', index=False)
print(f"✓ Lasso回归预测结果已保存到 lasso_predictions.csv")

# 3. 岭回归
print("训练岭回归...")
ridge_params = {'alpha': [0.001, 0.01, 0.1, 1, 10, 100]}
ridge_grid = GridSearchCV(Ridge(random_state=111), ridge_params, 
                         cv=6, scoring='neg_mean_squared_error', n_jobs=-1)
ridge_grid.fit(X_train_scaled, y_train_log)
ridge_model = ridge_grid.best_estimator_
ridge_results, ridge_pred, ridge_cv_scores = evaluate_model_with_cv(
    ridge_model, X_train_scaled, y_train_log, X_val_scaled, y_val_log, "Ridge"
)
models_results.append(ridge_results)
print(f"  最佳alpha: {ridge_grid.best_params_['alpha']}")
print(f"  RMSE: {ridge_results['val_rmse']:.4f}, R²: {ridge_results['val_r2']:.4f}")
print(f"  交叉验证R²: {ridge_results['cv_r2_mean']:.4f} ± {ridge_results['cv_r2_std']:.4f}")

# 输出岭回归预测结果
ridge_test_pred_log = ridge_model.predict(X_test_scaled)
ridge_test_pred = np.expm1(ridge_test_pred_log)
ridge_submission = pd.DataFrame({
    'ID': range(len(ridge_test_pred)),
    'Price': ridge_test_pred
})
ridge_submission.to_csv('ridge_predictions.csv', index=False)
print(f"✓ 岭回归预测结果已保存到 ridge_predictions.csv")

# 4. 弹性网络
print("训练弹性网络...")
elastic_params = {
    'alpha': [0.001, 0.01, 0.1, 1, 10],
    'l1_ratio': [0.1, 0.3, 0.5, 0.7, 0.9]
}
elastic_grid = GridSearchCV(ElasticNet(random_state=111, max_iter=1000), elastic_params, 
                           cv=6, scoring='neg_mean_squared_error', n_jobs=-1)
elastic_grid.fit(X_train_scaled, y_train_log)
elastic_model = elastic_grid.best_estimator_
elastic_results, elastic_pred, elastic_cv_scores = evaluate_model_with_cv(
    elastic_model, X_train_scaled, y_train_log, X_val_scaled, y_val_log, "Elastic Net"
)
models_results.append(elastic_results)
print(f"  最佳参数: alpha={elastic_grid.best_params_['alpha']}, l1_ratio={elastic_grid.best_params_['l1_ratio']}")
print(f"  RMSE: {elastic_results['val_rmse']:.4f}, R²: {elastic_results['val_r2']:.4f}")
print(f"  交叉验证R²: {elastic_results['cv_r2_mean']:.4f} ± {elastic_results['cv_r2_std']:.4f}")

# 输出弹性网络预测结果
elastic_test_pred_log = elastic_model.predict(X_test_scaled)
elastic_test_pred = np.expm1(elastic_test_pred_log)
elastic_submission = pd.DataFrame({
    'ID': range(len(elastic_test_pred)),
    'Price': elastic_test_pred
})
elastic_submission.to_csv('elastic_net_predictions.csv', index=False)
print(f"✓ 弹性网络预测结果已保存到 elastic_net_predictions.csv")

print("\n--- 模型比较和分析 ---")

# 创建结果比较表格
results_df = pd.DataFrame(models_results)

# 🔥 新增：添加Kaggle得分（根据您提供的信息）
kaggle_scores = {
    'Linear Regression': 65.1,
    'Lasso': 65.7,
    'Ridge': 65.7,
    'Elastic Net': 65.8
}

results_df['kaggle_score'] = results_df['model_name'].map(kaggle_scores)

# 按验证集RMSE排序
results_df = results_df.sort_values('val_rmse')

print("\n" + "="*100)
print("房价模型综合性能比较")
print("="*100)

# 🔥 新增：生成详细的性能表格
performance_table = results_df[[
    'model_name', 'train_r2', 'val_r2', 'cv_r2_mean', 'kaggle_score', 
    'train_rmse', 'val_rmse', 'val_mae'
]].round(4)

# 重命名列名以便更好理解
performance_table = performance_table.rename(columns={
    'model_name': '模型名称',
    'train_r2': '样本内R²',
    'val_r2': '样本外R²', 
    'cv_r2_mean': '交叉验证R²',
    'kaggle_score': 'Kaggle得分',
    'train_rmse': '训练集RMSE',
    'val_rmse': '验证集RMSE',
    'val_mae': '验证集MAE'
})

print(performance_table.to_string(index=False))

# 🔥 新增：生成总结表格（只包含R²相关指标）
print("\n" + "="*80)
print("R²指标对比总结")
print("="*80)

summary_table = results_df[[
    'model_name', 'train_r2', 'val_r2', 'cv_r2_mean', 'kaggle_score'
]].round(4)

summary_table = summary_table.rename(columns={
    'model_name': '模型',
    'train_r2': '样本内R²',
    'val_r2': '样本外R²',
    'cv_r2_mean': '交叉验证R²', 
    'kaggle_score': 'Kaggle得分'
})

print(summary_table.to_string(index=False))

# 选择最佳模型
best_model_info = results_df.iloc[0]
best_model_name = best_model_info['model_name']
best_val_rmse = best_model_info['val_rmse']

print(f"\n🏆 最佳模型: {best_model_name}")
print(f"最佳验证集RMSE: {best_val_rmse:.4f}")

print("\n" + "="*80)
print("模型训练总结")
print("="*80)
print(f"1. 数据集: 训练集 {X_train_final_cleaned.shape}, 验证集 {X_val_final_cleaned.shape}")
print(f"2. 特征数量: {X_train_final_cleaned.shape[1]}")
print(f"3. 模型比较: 测试了 {len(models_results)} 个模型")
print(f"4. 最佳模型: {best_model_name}")
print(f"5. 最佳验证集RMSE: {best_val_rmse:.4f} (对数尺度)")
print(f"6. 预测结果: 已生成4个CSV文件")
print(f"7. 交叉验证: 使用{results_df.iloc[0]['cv_folds']}折交叉验证")

# 🔥 新增：保存性能表格到CSV
performance_table.to_csv('model_performance_comparison.csv', index=False, encoding='utf-8-sig')
print(f"8. 性能对比表格已保存到: model_performance_comparison.csv")


开始模型训练和优化...
✓ 数据标准化完成

--- 线性模型 ---
训练线性回归...
  RMSE: 0.2624, R²: 0.8983
  交叉验证R²: 0.8996 ± 0.0021
✓ 线性回归预测结果已保存到 linear_regression_predictions.csv
训练Lasso回归...
  最佳alpha: 0.001
  RMSE: 0.2633, R²: 0.8977
  交叉验证R²: 0.8992 ± 0.0021
✓ Lasso回归预测结果已保存到 lasso_predictions.csv
训练岭回归...
  最佳alpha: 1
  RMSE: 0.2624, R²: 0.8983
  交叉验证R²: 0.8996 ± 0.0021
✓ 岭回归预测结果已保存到 ridge_predictions.csv
训练弹性网络...
  最佳参数: alpha=0.001, l1_ratio=0.1
  RMSE: 0.2625, R²: 0.8983
  交叉验证R²: 0.8996 ± 0.0021
✓ 弹性网络预测结果已保存到 elastic_net_predictions.csv

--- 模型比较和分析 ---

房价模型综合性能比较
             模型名称  样本内R²  样本外R²  交叉验证R²  Kaggle得分  训练集RMSE  验证集RMSE  验证集MAE
            Ridge 0.8998 0.8983  0.8996      65.7   0.2633   0.2624  0.1988
Linear Regression 0.8998 0.8983  0.8996      65.1   0.2633   0.2624  0.1988
      Elastic Net 0.8997 0.8983  0.8996      65.8   0.2634   0.2625  0.1990
            Lasso 0.8993 0.8977  0.8992      65.7   0.2639   0.2633  0.1995

R²指标对比总结
               模型  样本内R²  样本外R²  交叉验证R²  Kaggle得分
       